# PiGuard: lightweight DGA detector training

This notebook trains an **alert-only** classifier for domains that resemble malware-generated domains. It compares logistic regression and random forest, chooses a decision threshold on a validation set, evaluates once on a held-out test set, and exports a versioned deployment bundle for Raspberry Pi 3B+.

A high score means *suspicious-looking domain*, not *confirmed malware*. The default dataset has no malware-family labels, so this notebook does **not** claim unseen-family generalisation.

## 1. Setup

Upload `piguard_dga_features.py` alongside this notebook. Colab does not need a GPU for these models.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import sys
import urllib.request

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report, confusion_matrix,
    precision_recall_curve, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
MAX_FALSE_POSITIVE_RATE = 0.01
DATASET_URL = 'https://raw.githubusercontent.com/tisage/DGA-dataset/main/domains_features.gz'
WORK_DIR = Path('/content/piguard_dga') if Path('/content').exists() else Path('piguard_dga_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
print({'python': platform.python_version(), 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__, 'joblib': joblib.__version__})

In [ ]:
try:
    from piguard_dga_features import FEATURE_NAMES, FEATURE_VERSION, feature_frame, normalize_domain
except ModuleNotFoundError:
    if 'google.colab' not in sys.modules:
        raise
    from google.colab import files
    print('Upload piguard_dga_features.py')
    files.upload()
    from piguard_dga_features import FEATURE_NAMES, FEATURE_VERSION, feature_frame, normalize_domain

print(f'Feature contract {FEATURE_VERSION}: {len(FEATURE_NAMES)} features')

## 2. Download and validate the baseline dataset

The repository describes a roughly 100,000-domain dataset published with *A dictionary-based method for detecting machine-generated domains*. The currently downloaded file contains 91,599 rows, so the notebook records its SHA-256 hash and prints the actual class counts instead of relying on the repository summary. We recompute every feature so the training and Raspberry Pi inference paths use the same feature contract.

In [ ]:
dataset_path = WORK_DIR / 'domains_features.gz'
if not dataset_path.exists():
    urllib.request.urlretrieve(DATASET_URL, dataset_path)

dataset_sha256 = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
raw = pd.read_csv(dataset_path, compression='gzip', encoding='utf-8')
print('SHA-256:', dataset_sha256)
print('Shape:', raw.shape)
print('Columns:', raw.columns.tolist())
raw.head()

In [ ]:
def normalize_binary_label(value):
    if isinstance(value, (int, np.integer)) and int(value) in (0, 1):
        return int(value)
    if isinstance(value, (float, np.floating)) and float(value) in (0.0, 1.0):
        return int(value)
    text = str(value).strip().lower()
    if text in {'1', 'dga', 'malicious', 'positive', 'bad'}:
        return 1
    if text in {'0', 'benign', 'legit', 'legitimate', 'negative', 'good'}:
        return 0
    raise ValueError(f'Unrecognised label: {value!r}')

domain_column = next((c for c in raw.columns if c.lower() in {'domain', 'hostname', 'fqdn'}), None)
label_column = next((c for c in raw.columns if c.lower() in {'class', 'label', 'target', 'threat'}), None)
if domain_column is None or label_column is None:
    raise ValueError('Dataset must contain domain and binary label columns')

data = pd.DataFrame({
    'domain': raw[domain_column].map(normalize_domain),
    'label': raw[label_column].map(normalize_binary_label),
})
data = data[data['domain'].str.len().between(1, 253)].copy()

conflicting = data.groupby('domain')['label'].nunique()
conflicting_domains = set(conflicting[conflicting > 1].index)
data = data[~data['domain'].isin(conflicting_domains)].drop_duplicates('domain').reset_index(drop=True)

print('Clean shape:', data.shape)
print('Removed conflicting domains:', len(conflicting_domains))
print(data['label'].value_counts().rename(index={0: 'benign', 1: 'dga'}))

## 3. Split before model selection

The test set stays untouched until the model and threshold are selected. Because the default dataset has no family column, this is an in-dataset stratified split. Add a family-labelled dataset later and hold out complete DGA families for a stronger experiment.

In [ ]:
train_data, holdout_data = train_test_split(
    data, test_size=0.30, stratify=data['label'], random_state=SEED
)
validation_data, test_data = train_test_split(
    holdout_data, test_size=0.50, stratify=holdout_data['label'], random_state=SEED
)

for name, frame in [('train', train_data), ('validation', validation_data), ('test', test_data)]:
    print(name, len(frame), frame['label'].value_counts(normalize=True).sort_index().round(4).to_dict())

In [ ]:
X_train = feature_frame(train_data['domain'])
X_validation = feature_frame(validation_data['domain'])
X_test = feature_frame(test_data['domain'])
y_train = train_data['label'].to_numpy()
y_validation = validation_data['label'].to_numpy()
y_test = test_data['label'].to_numpy()

assert list(X_train.columns) == FEATURE_NAMES
assert np.isfinite(X_train.to_numpy()).all()
X_train.describe().T

## 4. Train explainable baseline models

In [ ]:
models = {
    'logistic_regression': Pipeline([
        ('scale', StandardScaler()),
        ('classifier', LogisticRegression(
            max_iter=2000, class_weight='balanced', solver='liblinear', random_state=SEED
        )),
    ]),
    'random_forest': RandomForestClassifier(
        n_estimators=250, max_depth=20, min_samples_leaf=2,
        class_weight='balanced_subsample', n_jobs=-1, random_state=SEED
    ),
}

for name, model in models.items():
    print('Training', name)
    model.fit(X_train, y_train)

## 5. Select the decision threshold on validation data

The threshold search prioritises recall while keeping the measured validation false-positive rate at or below 1%. That 1% is a development target, not a guarantee on a home network.

In [ ]:
def binary_metrics(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return {
        'threshold': float(threshold),
        'precision': float(precision_score(y_true, prediction, zero_division=0)),
        'recall': float(recall_score(y_true, prediction, zero_division=0)),
        'f1': float(f1_score(y_true, prediction, zero_division=0)),
        'false_positive_rate': float(fpr),
        'roc_auc': float(roc_auc_score(y_true, probability)),
        'confusion_matrix': [[int(tn), int(fp)], [int(fn), int(tp)]],
    }

def select_threshold(y_true, probability, max_fpr=0.01):
    candidates = []
    for threshold in np.linspace(0.01, 1.0, 200):
        metrics = binary_metrics(y_true, probability, threshold)
        if metrics['false_positive_rate'] <= max_fpr:
            candidates.append(metrics)
    if not candidates:
        raise RuntimeError('No threshold satisfies the false-positive constraint')
    return max(candidates, key=lambda item: (item['recall'], item['f1'], item['threshold']))

validation_results = {}
for name, model in models.items():
    probability = model.predict_proba(X_validation)[:, 1]
    validation_results[name] = select_threshold(y_validation, probability, MAX_FALSE_POSITIVE_RATE)

pd.DataFrame(validation_results).T

In [ ]:
selected_name = max(
    validation_results,
    key=lambda name: (validation_results[name]['f1'], validation_results[name]['recall'])
)
selected_model = models[selected_name]
selected_threshold = validation_results[selected_name]['threshold']
print('Selected model:', selected_name)
print('Selected threshold:', selected_threshold)

## 6. Final evaluation on the untouched test set

In [ ]:
test_probability = selected_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= selected_threshold).astype(int)
test_metrics = binary_metrics(y_test, test_probability, selected_threshold)
print(json.dumps(test_metrics, indent=2))
print(classification_report(y_test, test_prediction, target_names=['benign', 'dga'], digits=4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, display_labels=['benign', 'dga'], cmap='Blues', ax=axes[0], colorbar=False
)
axes[0].set_title('Held-out test confusion matrix')

fpr, tpr, _ = roc_curve(y_test, test_probability)
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['roc_auc']:.4f}")
axes[1].plot([0, 1], [0, 1], '--', color='grey')
axes[1].set(xlabel='False-positive rate', ylabel='True-positive rate', title='ROC curve')
axes[1].legend()

precision, recall, _ = precision_recall_curve(y_test, test_probability)
axes[2].plot(recall, precision)
axes[2].set(xlabel='Recall', ylabel='Precision', title='Precision-recall curve')
plt.tight_layout()
plt.show()

In [ ]:
if selected_name == 'random_forest':
    importance = pd.Series(selected_model.feature_importances_, index=FEATURE_NAMES).sort_values()
else:
    coefficients = selected_model.named_steps['classifier'].coef_[0]
    importance = pd.Series(np.abs(coefficients), index=FEATURE_NAMES).sort_values()
importance.plot.barh(figsize=(8, 6), title='Model feature importance')
plt.xlabel('Relative importance')
plt.tight_layout()
plt.show()

## 7. Optional local-benign validation

Do not upload identifiable household DNS logs to Colab. If you have an anonymised CSV containing only a `domain` column, set the flag below to `True`. This test measures false alerts on data closer to deployment.

In [ ]:
RUN_ANONYMISED_BENIGN_CHECK = False
local_benign_result = None
if RUN_ANONYMISED_BENIGN_CHECK:
    if 'google.colab' not in sys.modules:
        raise RuntimeError('Set a local CSV path when running outside Colab')
    from google.colab import files
    uploaded = files.upload()
    upload_name = next(iter(uploaded))
    local_domains = pd.read_csv(upload_name)['domain'].dropna().astype(str).map(normalize_domain).drop_duplicates()
    local_probability = selected_model.predict_proba(feature_frame(local_domains))[:, 1]
    local_benign_result = {
        'domain_count': int(len(local_domains)),
        'alert_count': int((local_probability >= selected_threshold).sum()),
        'observed_false_alert_rate': float((local_probability >= selected_threshold).mean()),
    }
    print(local_benign_result)

## 8. Export the deployment bundle

The bundle records the exact feature order, threshold, dependency versions, dataset hash, metrics, and limitations. Keep this bundle with the report results.

In [ ]:
artifact_dir = WORK_DIR / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / 'piguard_dga_model.joblib'
joblib.dump(selected_model, model_path, compress=3)
model_sha256 = hashlib.sha256(model_path.read_bytes()).hexdigest()

contract = {
    'project': 'PiGuard',
    'task': 'binary_dga_domain_classification',
    'positive_class': 'dga_like',
    'model_name': selected_name,
    'decision_threshold': float(selected_threshold),
    'feature_version': FEATURE_VERSION,
    'feature_names': FEATURE_NAMES,
    'dataset_url': DATASET_URL,
    'dataset_sha256': dataset_sha256,
    'model_sha256': model_sha256,
    'python_version': platform.python_version(),
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
    'scikit_learn_version': sklearn.__version__,
    'joblib_version': joblib.__version__,
    'random_seed': SEED,
    'maximum_validation_false_positive_rate': MAX_FALSE_POSITIVE_RATE,
    'deployment_mode': 'alert_only',
    'limitations': [
        'A high score indicates a DGA-like domain and does not prove malware infection.',
        'The default dataset has no DGA-family labels; the test split does not prove unseen-family generalisation.',
        'Deployment false-positive rate must be measured on local benign Pi-hole traffic.',
        'The model uses the likely registrable label only and does not inspect packet payloads or encrypted traffic.',
        'The offline multi-label public-suffix fallback is intentionally limited; production should use a maintained Public Suffix List.',
    ],
}
(artifact_dir / 'feature_contract.json').write_text(json.dumps(contract, indent=2), encoding='utf-8')

results = {
    'validation_results': validation_results,
    'selected_model': selected_name,
    'test_metrics': test_metrics,
    'test_classification_report': classification_report(
        y_test, test_prediction, target_names=['benign', 'dga'], output_dict=True, zero_division=0
    ),
    'split_sizes': {
        'train': int(len(train_data)), 'validation': int(len(validation_data)), 'test': int(len(test_data))
    },
    'optional_local_benign_check': local_benign_result,
}
(artifact_dir / 'metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')

requirements = (
    f'joblib=={joblib.__version__}\n'
    f'numpy=={np.__version__}\n'
    f'pandas=={pd.__version__}\n'
    f'scikit-learn=={sklearn.__version__}\n'
)
(artifact_dir / 'requirements-pi.txt').write_text(requirements, encoding='utf-8')

vector_sample = test_data.sample(n=min(20, len(test_data)), random_state=SEED).copy()
vector_features = feature_frame(vector_sample['domain'])
vector_probability = selected_model.predict_proba(vector_features)[:, 1]
test_vectors = [
    {
        'domain': domain, 'label': int(label),
        'expected_probability': float(probability),
        'expected_alert': bool(probability >= selected_threshold),
    }
    for domain, label, probability in zip(vector_sample['domain'], vector_sample['label'], vector_probability)
]
(artifact_dir / 'test_vectors.json').write_text(json.dumps(test_vectors, indent=2), encoding='utf-8')

zip_path = Path(shutil.make_archive(str(WORK_DIR / 'piguard_dga_artifacts'), 'zip', root_dir=WORK_DIR, base_dir='artifacts'))
print('Created:', zip_path)
print('Model size:', model_path.stat().st_size, 'bytes')
print('Model SHA-256:', model_sha256)

In [ ]:
sample_domains = ['google.com', 'xj3kq9z7p2.biz', 'www.wikipedia.org']
sample_probability = selected_model.predict_proba(feature_frame(sample_domains))[:, 1]
pd.DataFrame({
    'domain': sample_domains,
    'dga_probability': sample_probability,
    'alert': sample_probability >= selected_threshold,
})

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(zip_path))
else:
    print('Artifact bundle:', zip_path.resolve())

## Required follow-up on the Raspberry Pi

1. Install the exact exported dependency versions on 64-bit Raspberry Pi OS.
2. Run the supplied test vectors and confirm the probabilities match within a small tolerance.
3. Measure inference latency, peak RAM, and CPU usage.
4. Measure false alerts on several days of normal Pi-hole queries.
5. Keep the system in alert-only mode until those results are acceptable.